# Inflation Forecasting with Machine Learning

This notebook demonstrates how to use scikit-learn to forecast inflation using various models, with a focus on **ElasticNet regression**.

## Data Source
- **Training Data**: FRED-MD and FRED-QD databases (Federal Reserve Bank of St. Louis)
- **Real-time Data**: DPCCRV1Q225SBEA (PCE inflation)

## Models
- **ElasticNet** (Primary): Combines L1 and L2 regularization
- Ridge: L2 regularization
- Lasso: L1 regularization
- Random Forest: Ensemble method
- Gradient Boosting: Sequential ensemble

In [ ]:
# Import necessary libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, '../src' if os.path.exists('../src') else 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import download_fred_data, load_and_preprocess_data
from models import InflationForecaster, compare_models
from visualizations import (plot_predictions, plot_model_comparison, 
                           plot_residuals, plot_feature_importance)

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

## Step 1: Download and Load Data

In [ ]:
# Download data from FRED (if not already downloaded)
# Note: This may fail in restricted network environments
# In that case, use the sample data generator

try:
    download_fred_data(save_path='../data' if os.path.exists('../data') else 'data')
except Exception as e:
    print(f"Download failed: {e}")
    print("Using existing data or generating sample data...")

In [ ]:
# Load and preprocess data
data_path = '../data/fred_qd.csv' if os.path.exists('../data/fred_qd.csv') else 'data/fred_qd.csv'

X, y, features = load_and_preprocess_data(data_path=data_path)

print(f"\nData Summary:")
print(f"  Shape: {X.shape}")
print(f"  Features: {len(features)}")
print(f"  Date range: {X.index.min()} to {X.index.max()}")
print(f"  Target variable range: {y.min():.2f} to {y.max():.2f}")

## Step 2: Exploratory Data Analysis

In [ ]:
# Plot target variable over time
plt.figure(figsize=(14, 6))
plt.plot(y.index, y.values, linewidth=2)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Target Variable (CPI)', fontsize=12)
plt.title('Target Variable Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Check for missing values
missing_pct = (X.isna().sum() / len(X) * 100).sort_values(ascending=False)
print("Features with missing values:")
print(missing_pct[missing_pct > 0])

if missing_pct.max() == 0:
    print("✓ No missing values in features")

In [ ]:
# Show correlation of top features with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("\nTop 10 features correlated with target:")
print(correlations.head(10))

# Plot
plt.figure(figsize=(10, 6))
correlations.head(15).plot(kind='barh', color='steelblue', alpha=0.7)
plt.xlabel('Absolute Correlation with Target', fontsize=12)
plt.title('Top 15 Features by Correlation with Target', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 3: Train ElasticNet Model (Primary Model)

In [ ]:
# Split data (preserve temporal order)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set: {len(X_train)} samples ({X_train.index.min()} to {X_train.index.max()})")
print(f"Test set: {len(X_test)} samples ({X_test.index.min()} to {X_test.index.max()})")

In [ ]:
# Train ElasticNet with hyperparameter tuning
elasticnet = InflationForecaster(model_type='elasticnet', random_state=42)
elasticnet.train(X_train, y_train, tune_hyperparameters=True, cv_folds=3)

# Evaluate
metrics, y_pred = elasticnet.evaluate(X_test, y_test)

print(f"\nElasticNet Performance:")
print(f"  RMSE: {metrics['rmse']:.4f}")
print(f"  MAE:  {metrics['mae']:.4f}")
print(f"  R²:   {metrics['r2']:.4f}")
print(f"  MAPE: {metrics['mape']:.2f}%")
print(f"\nBest parameters: {elasticnet.best_params}")

In [ ]:
# Plot predictions vs actual
plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values, 'o-', label='Actual', linewidth=2, markersize=6)
plt.plot(y_test.index, y_pred, 's--', label='ElasticNet Prediction', linewidth=2, alpha=0.7)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Target Value', fontsize=12)
plt.title('ElasticNet: Actual vs Predicted', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance_df = elasticnet.get_feature_importance(features)

if importance_df is not None:
    print("\nTop 10 Most Important Features:")
    print(importance_df.head(10).to_string(index=False))
    
    # Plot
    plot_feature_importance(importance_df, top_n=15)

## Step 4: Compare Multiple Models

In [ ]:
# Compare all models
results_df, predictions, X_test_cmp, y_test_cmp = compare_models(X, y, test_size=0.2, random_state=42)

print("\nModel Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Plot model comparison
plot_model_comparison(results_df)

In [ ]:
# Plot predictions from all models
plot_predictions(y_test_cmp.values, predictions, dates=y_test_cmp.index)

## Step 5: Residual Analysis

In [ ]:
# Residual analysis for ElasticNet
plot_residuals(y_test.values, y_pred, model_name='ElasticNet')

## Step 6: Model Interpretation

### ElasticNet Coefficients

ElasticNet combines L1 and L2 regularization:
- **L1 (Lasso)**: Encourages sparsity, some coefficients become exactly zero
- **L2 (Ridge)**: Shrinks coefficients but keeps all features
- **l1_ratio**: Controls the balance (0 = pure Ridge, 1 = pure Lasso)

In [ ]:
# Show which features have non-zero coefficients
coefs = pd.DataFrame({
    'feature': features,
    'coefficient': elasticnet.model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

non_zero = coefs[coefs['coefficient'] != 0]
print(f"\nFeatures with non-zero coefficients: {len(non_zero)} out of {len(features)}")
print("\nTop 10 by absolute coefficient value:")
print(non_zero.head(10).to_string(index=False))

## Conclusion

This notebook demonstrated:
1. Loading and preprocessing FRED economic data
2. Training an ElasticNet model for inflation forecasting
3. Comparing multiple regression models
4. Evaluating model performance and interpreting results

### Key Findings:
- ElasticNet successfully balances model complexity and predictive power
- Feature selection through L1 regularization helps identify key economic indicators
- Time series cross-validation ensures robust hyperparameter tuning

### Next Steps:
- Try different target variables (e.g., inflation rate instead of price level)
- Experiment with feature engineering (lags, differences, interactions)
- Test on out-of-sample data with real-time FRED API
- Consider more advanced models (LSTM, XGBoost, etc.)